In [ ]:
import sys
import time
import os
import json
from pathlib import Path
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from src.utils.elastic_client import ElasticSearchClient
from src.utils.data_processor import chunking_by_sentences
from src.utils.data_processor import embed, generate_stream
from src.utils.neo4j_client import Neo4jClient
from underthesea import sent_tokenize
from dotenv import load_dotenv

load_dotenv()

In [2]:
ELASTIC_HOST = os.environ.get("ELASTIC_HOST")
API_KEY = os.environ.get('GENAI_API')
URI = os.environ.get('URI')
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD")

AUTH = (user, password)

In [ ]:
# ===== CONFIGURATION =====
BATCH_SIZE = 100  # Documents per batch (adjust based on memory/API limits)
EMBEDDING_BATCH_SIZE = 50  # Chunks to embed per API call
RPM_LIMIT = 60
SLEEP_PER_REQUEST = 60 / RPM_LIMIT
CHECKPOINT_FILE = "/tmp/rag_checkpoint.json"

def load_checkpoint():
    """Load last checkpoint from file"""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            print(f"Loaded checkpoint: post_id={data['last_checkpoint']}, processed={data['total_count']}")
            return data['last_checkpoint'], data['total_count']
    return 0, 0

def save_checkpoint(last_checkpoint, total_count):
    """Save checkpoint to file"""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_checkpoint': last_checkpoint, 'total_count': total_count}, f)
    print(f"Saved checkpoint: post_id={last_checkpoint}, total_processed={total_count}")

### Stepback prompting

In [7]:
stepback_system_message = f"""
Bạn là một chuyên gia phân tích dữ liệu truyền thông và báo chí. Nhiệm vụ của bạn là áp dụng kỹ thuật "Bước lùi ngữ cảnh" (Step-Back Prompting) để hỗ trợ hệ thống tra cứu bài báo. 

Khi nhận được một câu hỏi chi tiết về một tình tiết, con số hoặc phát ngôn cụ thể trong một bài báo, bạn phải lùi lại một bước để tạo ra một câu hỏi tổng quan hơn (Step-back question) về dòng sự kiện, chủ đề, hoặc bối cảnh vĩ mô của thực thể đó. Câu hỏi tổng quan này giúp hệ thống tìm kiếm trọn vẹn toàn bộ bài báo chứa ngữ cảnh thay vì bị mắc kẹt vào các từ khóa vụn vặt.

Hãy nghiên cứu kỹ các ví dụ mẫu (Few-shot) về báo chí dưới đây:

---
Ví dụ 1:
"input": "Giá vàng SJC vào lúc 9h sáng nay tăng bao nhiêu tiền một lượng?"
"output": "Diễn biến xu hướng giá vàng SJC và thị trường kim loại quý trong nước thời gian gần đây như thế nào?"

Ví dụ 2:
"input": "Tên của chị gái bà Lan (đối tượng bị bắt trong vụ án) là gì?"
"output": "Thông tin về lý lịch nhân thân, gia đình và các đồng phạm liên quan đến mạng lưới của bà Lan trong vụ án là gì?"

Ví dụ 3:
"input": "Ông Nguyễn Văn A đã phát biểu những gì tại hội nghị chuyển đổi số hôm qua?"
"output": "Nội dung chính, các thông điệp cốt lõi và diễn biến tổng thể của hội nghị chuyển đổi số vừa diễn ra là gì?"

Ví dụ 4:
"input": "Cổ phiếu của công ty X bị giảm sàn bao nhiêu phiên sau scandal?"
"output": "Lịch sử biến động thị giá của cổ phiếu công ty X và phản ứng của thị trường chứng khoán trước các thông tin tiêu cực là gì?"

Ví dụ 5:
"input": "Vụ tai nạn ở cao tốc có bao nhiêu người bị thương?"
"output": "Thông tin tổng hợp về thiệt hại, nguyên nhân và công tác khắc phục hậu quả của vụ tai nạn giao thông trên cao tốc là gì?"
---

Bây giờ, hãy phân tích câu hỏi báo chí dưới đây và tạo ra câu hỏi bước lùi chuẩn xác
"""

user_message = "Tên của chị Lan là gì"

generate_stream(user_message, stepback_system_message, model='gemini-2.5-flash-lite')

Thông tin tổng hợp về nhân thân, vai trò và sự liên quan của chị Lan trong sự kiện hoặc bối cảnh được đề cập là gì?

### Parent document retriever

In [3]:
es = ElasticSearchClient(ELASTIC_HOST)
es.get_info()

ObjectApiResponse({'name': 'e788d505d845', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'MFtuXAmyTy2S3RIzHPzhSA', 'version': {'number': '8.13.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'da95df118650b55a500dcc181889ac35c6d8da7c', 'build_date': '2024-05-06T22:04:45.107454559Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [4]:
doc = es.get_by_id(index_name='cafef-xahoi', doc_id='188260516203640785')
doc

ObjectApiResponse({'_index': 'cafef-xahoi', '_id': '188260516203640785', '_version': 1, '_seq_no': 3, '_primary_term': 1, 'found': True, '_source': {'title': 'Thêm một loại giấy tờ sẽ được tích hợp trên VNeID,   người dân cả nước cần nắm rõ', 'detail_sapo': 'Từ 1/7 tới đây, sẽ có đổi mới liên quan đến việc sử dụng ứng dụng VNeID mà người dân cần biết.', 'author': 'Theo Huỳnh Duy', 'published_at': '2026-05-17-09-34', 'content': 'TIN MỚI\nBộ Công an đang xây dựng dự thảo thông tư quy định về sử dụng biểu mẫu, quản lý và khai thác cơ sở dữ liệu lý lịch tư pháp; đồng thời hướng dẫn trình tự, thủ tục cấp phiếu lý lịch tư pháp và cung cấp thông tin lý lịch tư pháp.\nTheo dự thảo, từ ngày 1/7/2026, thông tin lý lịch tư pháp sẽ được hiển thị trên ứng dụng VNeID đối với các hồ sơ yêu cầu cấp phiếu qua VNeID và Cổng Dịch vụ công quốc gia.\nNội dung hiển thị gồm thông tin chung về tình trạng án tích, thời gian cập nhật dữ liệu gần nhất và thông tin chi tiết tương ứng với nội dung trên phiếu lý lị

In [5]:
def clean_and_split_paragraphs(content: str, min_words: int = 40) -> list[str]:
    """
    Tách đoạn văn lớn (Parent) nhưng bảo toàn nguyên vẹn cấu trúc câu.
    Gom các câu liên tiếp lại cho đến khi tổng số từ đạt ngưỡng min_words mới ngắt đoạn.
    
    :param content: Văn bản thô ghép từ Title + Sapo + Content
    :param min_words: Số từ tối thiểu cho một Parent Node (Nên tăng lên ~40-50 từ 
                      để tương đương một đoạn văn chuẩn)
    """
    if not content.strip():
        return []
        
    sentences = sent_tokenize(content)
    
    refined_paragraphs = []
    current_paragraph_sentences = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        sentence_word_count = len(sentence.split())
        
        current_paragraph_sentences.append(sentence)
        current_word_count += sentence_word_count
        
        # Nếu tổng số từ của các câu tích lũy ĐÃ ĐỦ lớn (>= min_words)
        # Thì đóng gói đoạn đó lại và chuyển sang đoạn mới
        if current_word_count >= min_words:
            refined_paragraphs.append(" ".join(current_paragraph_sentences))
            # Reset để gom đoạn tiếp theo
            current_paragraph_sentences = []
            current_word_count = 0
            
    # Xử lý đoạn cuối cùng còn sót lại (nếu có câu chưa được đóng gói)
    if current_paragraph_sentences:
        # Nếu đoạn cuối quá ngắn và trước đó đã có đoạn khác, gộp luôn vào đoạn trước cho sạch dữ liệu
        if refined_paragraphs and current_word_count < (min_words / 2):
            refined_paragraphs[-1] += " " + " ".join(current_paragraph_sentences)
        else:
            refined_paragraphs.append(" ".join(current_paragraph_sentences))
            
    return refined_paragraphs

In [ ]:
CYPHER_INSERT_HIERARCHY = """
// Bước 1: Khởi tạo/Cập nhật Node gốc (Sử dụng MERGE để tránh trùng lặp)
MERGE (pdf:PDFNode {id: $pdf_id})
SET pdf.title = $title,
    pdf.post_url = $post_url,
    pdf.published_at = $published_at,
    pdf.author = $author

// Bước 2: Tách riêng phần xóa cấu trúc cũ, chạy chớp nhoáng
WITH pdf
OPTIONAL MATCH (pdf)-[:HAS_PARENT]->(p:ParentDocumentNode)
OPTIONAL MATCH (p)-[:HAS_CHILD]->(c:ChildDocumentNode)
DETACH DELETE p, c

// Bước 3: Đưa về 1 dòng duy nhất sau khi dọn dẹp
WITH DISTINCT pdf

// Bước 4: Tạo toàn bộ các Node Parent cùng một lúc
UNWIND $parent_nodes AS p_data
MERGE (p:ParentDocumentNode {id: p_data.parent_id})
SET p.parent_text = p_data.parent_text
MERGE (pdf)-[:HAS_PARENT]->(p)

// Bước 5: Làm phẳng mảng Child để đẩy một mẻ duy nhất, không dùng CALL lồng dòng nữa
WITH DISTINCT pdf, p_data, p
UNWIND p_data.child_nodes AS c_data
MERGE (c:ChildDocumentNode {id: c_data.child_id})
SET c.text = c_data.child_text,
    c.embedding = c_data.embedding
MERGE (p)-[:HAS_CHILD]->(c)
"""

def ingest_es_record_to_graph(record):
    source = record["_source"]
    pdf_id = record["_id"]
    
    content = '. '.join([source['title'], source['detail_sapo'], source['content']])
    paragraphs = clean_and_split_paragraphs(content, min_words=30)
   
    all_child_texts = []
    structure_mapping = [] 
    
    for p_idx, para in enumerate(paragraphs):
        parent_id = f"{pdf_id}_p_{p_idx}"
        child_chunks = chunking_by_sentences(para, chunk_size=1, overlap=0)
        
        structure_mapping.append({
            "parent_id": parent_id,
            "parent_text": para,
            "child_count": len(child_chunks)
        })
        all_child_texts.extend(child_chunks)
        
    if not all_child_texts:
        return

    all_embeddings = embed(all_child_texts) 
  
    parent_nodes_payload = []
    vector_idx = 0
    
    for p_info in structure_mapping:
        parent_id = p_info["parent_id"]
        child_nodes_payload = []
        
        for c_idx in range(p_info["child_count"]):
            child_nodes_payload.append({
                "child_id": f"{parent_id}_c_{c_idx}",
                "child_text": all_child_texts[vector_idx],
                "embedding": all_embeddings[vector_idx]
            })
            vector_idx += 1
            
        parent_nodes_payload.append({
            "parent_id": parent_id,
            "parent_text": p_info["parent_text"],
            "child_nodes": child_nodes_payload
        })
        
    params = {
        "pdf_id": pdf_id,
        "title": source["title"],
        "post_url": source["post_url"],
        "published_at": source["published_at"],
        "author": source["author"],
        "parent_nodes": parent_nodes_payload
    }


    with Neo4jClient(URI, AUTH) as session:
        session.query(CYPHER_INSERT_HIERARCHY, parameters=params)
        
    print(f"Đã nạp thành công bài báo ID {pdf_id} theo cấu trúc Parent-Child")

In [ ]:
def process_batch_with_batched_embeddings(batch_docs, embedding_batch_size=EMBEDDING_BATCH_SIZE):
    """
    Process a batch of ES documents and collect all chunks before embedding.
    This is the KEY optimization - embedding multiple documents' chunks together!
    
    Returns: list of (parent_nodes_payload, pdf_record) tuples
    """
    all_chunks_with_metadata = []  # (chunk_text, pdf_id, parent_id, parent_idx, child_idx)
    all_child_texts = []
    
    # Step 1: Extract all chunks from all documents in batch
    for doc in batch_docs:
        source = doc["_source"]
        pdf_id = doc["_id"]
        
        content = '. '.join([source['title'], source['detail_sapo'], source['content']])
        paragraphs = clean_and_split_paragraphs(content, min_words=30)
        
        for p_idx, para in enumerate(paragraphs):
            parent_id = f"{pdf_id}_p_{p_idx}"
            child_chunks = chunking_by_sentences(para, chunk_size=1, overlap=0)
            
            for c_idx, chunk in enumerate(child_chunks):
                all_chunks_with_metadata.append({
                    'text': chunk,
                    'pdf_id': pdf_id,
                    'parent_id': parent_id,
                    'parent_text': para,
                    'p_idx': p_idx,
                    'c_idx': c_idx
                })
                all_child_texts.append(chunk)
    
    if not all_child_texts:
        return []
    
    # Step 2: BATCH EMBEDDING - Process all chunks at once (or in chunks if too many)
    # This is where we save API calls!
    all_embeddings = []
    for i in range(0, len(all_child_texts), embedding_batch_size):
        chunk_batch = all_child_texts[i:i + embedding_batch_size]
        embeddings = embed(chunk_batch)  # ONE API call for 50 chunks instead of 1 call per chunk!
        all_embeddings.extend(embeddings)
        time.sleep(SLEEP_PER_REQUEST)
    
    # Step 3: Reconstruct hierarchical structure with embeddings
    results = []
    parent_nodes_by_doc = {}  # Group by pdf_id
    
    for idx, metadata in enumerate(all_chunks_with_metadata):
        pdf_id = metadata['pdf_id']
        parent_id = metadata['parent_id']
        
        if pdf_id not in parent_nodes_by_doc:
            parent_nodes_by_doc[pdf_id] = {}
        
        if parent_id not in parent_nodes_by_doc[pdf_id]:
            parent_nodes_by_doc[pdf_id][parent_id] = {
                'parent_id': parent_id,
                'parent_text': metadata['parent_text'],
                'child_nodes': [],
                'source': None
            }
        
        # Find original doc for source info
        if parent_nodes_by_doc[pdf_id][parent_id]['source'] is None:
            for doc in batch_docs:
                if doc['_id'] == pdf_id:
                    parent_nodes_by_doc[pdf_id][parent_id]['source'] = doc['_source']
                    break
        
        parent_nodes_by_doc[pdf_id][parent_id]['child_nodes'].append({
            'child_id': f"{parent_id}_c_{metadata['c_idx']}",
            'child_text': metadata['text'],
            'embedding': all_embeddings[idx]
        })
    
    # Flatten structure
    for pdf_id, parents in parent_nodes_by_doc.items():
        for parent_id, parent_data in parents.items():
            results.append((pdf_id, parent_data))
    
    return results

In [ ]:
count = 0
last_checkpoint, total_count = load_checkpoint()

es_stream = es.incremental_scan(
    index_name="cafef-xahoi",
    incremental_column="post_id",
    checkpoint_value=last_checkpoint,
    batch_size=500
)

for es_batch in es_stream:
    print(f"\n=== Fetched ES batch size={len(es_batch)} ===")
    
    # Process ES batch into smaller document batches (configurable)
    for doc_batch_idx in range(0, len(es_batch), BATCH_SIZE):
        doc_batch = es_batch[doc_batch_idx:doc_batch_idx + BATCH_SIZE]
        print(f"Processing sub-batch {doc_batch_idx//BATCH_SIZE + 1} with {len(doc_batch)} docs")
        
        batch_start = time.time()
        
        try:
            # KEY OPTIMIZATION: Process entire batch with BATCHED embeddings
            results = process_batch_with_batched_embeddings(doc_batch, embedding_batch_size=EMBEDDING_BATCH_SIZE)
            
            # Write to Neo4j
            for pdf_id, parent_data in results:
                source = parent_data['source']
                params = {
                    "pdf_id": pdf_id,
                    "title": source["title"],
                    "post_url": source["post_url"],
                    "published_at": source["published_at"],
                    "author": source["author"],
                    "parent_nodes": [parent_data]
                }
                
                with Neo4jClient(URI, AUTH) as session:
                    session.query(CYPHER_INSERT_HIERARCHY, parameters=params)
                
                total_count += 1
            
            # Update checkpoint after successful batch
            last_checkpoint = doc_batch[-1]["_source"]["post_id"]
            save_checkpoint(last_checkpoint, total_count)
            
            batch_time = time.time() - batch_start
            print(f"Batch done in {batch_time:.1f}s | Total processed: {total_count} | Checkpoint: {last_checkpoint}")
            
        except Exception as batch_error:
            print(f"ERROR in batch: {batch_error}")
            raise